# 3️⃣ ⚙️ Train locally, with data on the cloud !

## `make run_preprocess`

⚠️⚠️⚠️ The test `make test_preprocessing` may not work due to extra datetime column (8 != 7). Check test file! `tests/cloud_training/test_main.py`, line 53.

Set up the challenge:

1. `direnv allow`
2. Rename `.env.sample` to `.env`
3. Replace `GCP_PROJECT` with you own
4. `direnv reload`

### `preprocess()`

At this stage, you need to work on the `preprocess` function in `taxifare/interface/main`

This is how you find it 👇

In [ ]:
def preprocess(min_date:str = '2009-01-01', max_date:str = '2015-01-01') -> None:
    """
    - Query the raw dataset from Le Wagon's BigQuery dataset
    - Cache query result as a local CSV if it doesn't exist locally
    - Process query data
    - Store processed data on your personal BQ (truncate existing table if it exists)
    - No need to cache processed data as CSV (it will be cached when queried back from BQ during training)
    """

    print(Fore.MAGENTA + "\n ⭐️ Use case: preprocess" + Style.RESET_ALL)

    # Query raw data from BigQuery using `get_data_with_cache`
    min_date = parse(min_date).strftime('%Y-%m-%d') # e.g '2009-01-01'
    max_date = parse(max_date).strftime('%Y-%m-%d') # e.g '2009-01-01'

    query = f"""
        SELECT {",".join(COLUMN_NAMES_RAW)}
        FROM {GCP_PROJECT_WAGON}.{BQ_DATASET}.raw_{DATA_SIZE}
        WHERE pickup_datetime BETWEEN '{min_date}' AND '{max_date}'
        ORDER BY pickup_datetime
    """

    pass  # YOUR CODE HERE

    # Process data
    pass  # YOUR CODE HERE
    # Load a DataFrame onto BigQuery containing [pickup_datetime, X_processed, y]
    # using data.load_data_to_bq()
    pass  # YOUR CODE HERE

    print("✅ preprocess() done \n")

You'll need to use `taxifare.ml_logic.data`'s following functions:
- `get_data_with_cache()`
    - This function is ready and takes the query we give you as argument.
    - It loads data locally if it exists, if not load from cloud, and create local cache.
- `clean_data()`
    - Also ready, takes a DF and preprocesses it 
- `load_data_to_bq`
    - ⚠️ This function needs to be completed ⚠️( SEE NEXT SECTION )
    - It uploads the preprocessed df to bigquery
    
This is how `preprocess()` should become 👇

In [2]:
def preprocess(min_date:str = '2009-01-01', max_date:str = '2015-01-01') -> None:
    """
    - Query the raw dataset from Le Wagon's BigQuery dataset.
    - Cache query result as a local CSV if it doesn't exist locally.
    - Process query data.
    - Store processed data on your personal BQ (truncate existing table if it exists).
    - No need to cache processed data as CSV (it will be cached when queried back from BQ during training).
    """
    
    # Initial setup for printing and data handling
    print(Fore.MAGENTA + "\n ⭐️ Use case: preprocess" + Style.RESET_ALL)

    # Convert min_date and max_date strings to a specific format ('YYYY-MM-DD').
    min_date = parse(min_date).strftime('%Y-%m-%d')
    max_date = parse(max_date).strftime('%Y-%m-%d')

    # Formulate a SQL query to select all columns between specified dates from a BigQuery dataset.
    query = f"""
        SELECT *
        FROM {GCP_PROJECT_WAGON}.{BQ_DATASET}.raw_{DATA_SIZE}
        WHERE pickup_datetime BETWEEN '{min_date}' AND '{max_date}'
        ORDER BY pickup_datetime
    """
    
    # Retrieve data, caching it locally if not already cached. This involves:
    # - Checking if the query result exists as a CSV file locally.
    # - If not, executing the query on BigQuery and saving the result as a CSV.
    data_query_cache_path = Path(LOCAL_DATA_PATH).joinpath("raw", f"query_{min_date}_{max_date}_{DATA_SIZE}.csv")
    data_query = get_data_with_cache(
        query=query,
        gcp_project=GCP_PROJECT,
        cache_path=data_query_cache_path,
        data_has_header=True
    )

    # Clean the data retrieved from BigQuery.
    data_clean = clean_data(data_query)

    # Prepare the features (X) and target variable (y) for machine learning models.
    X = data_clean.drop("fare_amount", axis=1)
    y = data_clean[["fare_amount"]]

    # Process features, possibly including normalization, encoding, etc.
    X_processed = preprocess_features(X)

    # Concatenate the cleaned pickup_datetime, processed features, and target variable into a single DataFrame.
    data_processed_with_timestamp = pd.DataFrame(np.concatenate((
        data_clean[["pickup_datetime"]],
        X_processed,
        y,
    ), axis=1))
    
    
    # ⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️
    #⚠️⚠️⚠️ NEED TO WORK ON THAT FUNCTION IN SEPARATE FILE ⚠️⚠️⚠️⚠️
    # Load the processed data into a BigQuery table, truncating the existing table if it exists.
    load_data_to_bq(
        data_processed_with_timestamp,
        gcp_project=GCP_PROJECT,
        bq_dataset=BQ_DATASET,
        table=f'processed_{DATA_SIZE}',
        truncate=True
    )
    #⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️

    # Confirm the process is complete.
    print("✅ preprocess() done \n")


<details>
<summary>Expand for a detailed breakdown of the preprocessing steps</summary>

- **Initial Setup for Printing and Data Handling:** The use of `Fore.MAGENTA` and `Style.RESET_ALL` is for making the console output colorful and more readable. This doesn't affect the data processing but improves user interaction.

- **Date Formatting:** The `min_date` and `max_date` are reformatted for consistency with the database's expected date format, ensuring the query operates correctly.

- **SQL Query Adjustment:** The query is adjusted to select all columns, which allows for more flexibility in processing different datasets without changing the column names manually.

- **Data Caching:** This step checks if the query result is already saved locally as a CSV file. If not, it executes the query on BigQuery and saves the result. This minimizes costs and speeds up the process by avoiding repeated queries for the same data.

- **Data Cleaning and Processing:** After retrieving the data, it undergoes cleaning (removing or correcting anomalies) and processing (preparing for machine learning models). This could involve handling missing values, encoding categorical variables, scaling numerical features, etc.

- **Preparing Data for Machine Learning:** The features (`X`) and target variable (`y`) are separated to facilitate machine learning model training. This is a critical step in data preparation.

- **Concatenating Processed Data:** The processed features, target variable, and original `pickup_datetime` are combined into a single DataFrame. This step is crucial for maintaining the relationship between the features and the target while also keeping track of the temporal aspect of the data.

- **Loading Data to BigQuery:** The final processed dataset is uploaded to BigQuery, with the option to truncate the existing table. This means the current table is deleted and replaced with

</details>


#### `load_data_to_bq()`

This is how to find `load_Data_bq()` 👇


In [ ]:
def load_data_to_bq(
        data: pd.DataFrame,
        gcp_project:str,
        bq_dataset:str,
        table: str,
        truncate: bool
    ) -> None:
    """
    - Save the DataFrame to BigQuery
    - Empty the table beforehand if `truncate` is True, append otherwise
    """

    assert isinstance(data, pd.DataFrame)
    full_table_name = f"{gcp_project}.{bq_dataset}.{table}"
    print(Fore.BLUE + f"\nSave data to BigQuery @ {full_table_name}...:" + Style.RESET_ALL)

    # Load data onto full_table_name

    # 🎯 HINT for "*** TypeError: expected bytes, int found":
    # After preprocessing the data, your original column names are gone (print it to check),
    # so ensure that your column names are *strings* that start with either 
    # a *letter* or an *underscore*, as BQ does not accept anything else

    pass  # YOUR CODE HERE

    print(f"✅ Data saved to bigquery, with shape {data.shape}")


and how it should end up looking like 👇

In [ ]:
def load_data_to_bq(
        data: pd.DataFrame,
        gcp_project: str,
        bq_dataset: str,
        table: str,
        truncate: bool
    ) -> None:
    """
    Save the DataFrame to BigQuery.
    Empty the table beforehand if `truncate` is True, append otherwise.

    Parameters:
    - data: The pandas DataFrame to save.
    - gcp_project: The Google Cloud Platform project ID.
    - bq_dataset: The BigQuery dataset name.
    - table: The BigQuery table name.
    - truncate: If True, the table is emptied before the data is loaded; if False, data is appended.
    """

    # Check that the input is indeed a pandas DataFrame.
    assert isinstance(data, pd.DataFrame), "Input data must be a pandas DataFrame."

    # Create a string that combines the project, dataset, and table names in a way BigQuery expects.
    full_table_name = f"{gcp_project}.{bq_dataset}.{table}"
    print(Fore.BLUE + f"\nSave data to BigQuery @ {full_table_name}...:" + Style.RESET_ALL)

    # Adjust column names to ensure they start with a letter or an underscore.
    # BigQuery requires this format for column names.
    data.columns = [
        f"_{column}" if not str(column)[0].isalpha() and not str(column)[0] == "_" else str(column)
        for column in data.columns
    ]

    # Create a BigQuery client to interact with the BigQuery service.
    client = bigquery.Client(project=gcp_project)

    # Set up the job configuration to either truncate the table or append data based on the 'truncate' parameter.
    write_mode = bigquery.WriteDisposition.WRITE_TRUNCATE if truncate else bigquery.WriteDisposition.WRITE_APPEND
    job_config = bigquery.LoadJobConfig(write_disposition=write_mode)

    # Inform the user about the operation being performed, based on the truncate flag.
    action = "Truncating and writing" if truncate else "Appending"
    print(f"\n{action} to {full_table_name} ({data.shape[0]} rows)")

    # Load the DataFrame to the specified BigQuery table.
    job = client.load_table_from_dataframe(data, full_table_name, job_config=job_config)
    result = job.result()  # Wait for the job to complete

    # Confirm the operation's success to the user.
    print(f"✅ Data successfully {'written to' if truncate else 'appended in'} BigQuery, with shape {data.shape}")


<details>
<summary><b>Explanation of the function steo by step:</b></summary>

1. **Parameter Documentation:** The function parameters are clearly documented, explaining what each parameter is for and how it affects the function's behavior.

2. **Assertion Check:** The code checks if the input `data` is a pandas DataFrame. This ensures that the function doesn't proceed with invalid input, which could lead to errors.

3. **Column Name Adjustment:** Before loading the data to BigQuery, the column names are adjusted to comply with BigQuery's naming requirements. BigQuery requires column names to start with a letter or an underscore. This step modifies any column names that don't meet this criterion, ensuring the data loading process doesn't fail due to invalid column names.

4. **BigQuery Client Initialization:** A BigQuery client is created using the `gcp_project` parameter. This client is used to interact with the BigQuery service, allowing the function to load data into the specified table.

5. **Job Configuration:** Depending on the `truncate` parameter, the function configures the load job to either truncate the existing table before loading the new data or append the new data to the existing table.

6. **Loading Data:** The DataFrame is loaded to the specified BigQuery table using the configured job settings. The process waits for completion, ensuring that the data is fully loaded before the function completes.

7. **Confirmation Message:** After the data is loaded, the function prints a confirmation message, indicating whether the data was written to a new table or appended to an existing one, along with the shape (number of rows and columns) of the data loaded.

This function is a comprehensive tool for handling the uploading of DataFrame data to Google BigQuery, providing flexibility through the `truncate` parameter and ensuring data integrity through column name adjustments.

</details>


At this stage, `make run_preprocessing` should work.

If you set `MODEL_TARGET=gcs` in the `.env`, and run `make run_preprocessing`, you should see a preprocessed table appear in your bigquery.

If it does, you're done for `make run_preprocessing` 🥳🥳🥳🥳

## `make run_train`

At this stage, you need to work on the `train` function in `taxifare/interface/main`.

You'll find it like this 👇

In [ ]:
def train(
        min_date:str = '2009-01-01',
        max_date:str = '2015-01-01',
        split_ratio: float = 0.02, # 0.02 represents ~ 1 month of validation data on a 2009-2015 train set
        learning_rate=0.0005,
        batch_size = 256,
        patience = 2
    ) -> float:

    """
    - Download processed data from your BQ table (or from cache if it exists)
    - Train on the preprocessed dataset (which should be ordered by date)
    - Store training results and model weights

    Return val_mae as a float
    """

    print(Fore.MAGENTA + "\n⭐️ Use case: train" + Style.RESET_ALL)
    print(Fore.BLUE + "\nLoading preprocessed validation data..." + Style.RESET_ALL)

    min_date = parse(min_date).strftime('%Y-%m-%d') # e.g '2009-01-01'
    max_date = parse(max_date).strftime('%Y-%m-%d') # e.g '2009-01-01'

    # Load processed data using `get_data_with_cache` in chronological order
    # Try it out manually on console.cloud.google.com first!

    pass  # YOUR CODE HERE

    # Create (X_train_processed, y_train, X_val_processed, y_val)
    pass  # YOUR CODE HERE

    # Train model using `model.py`
    pass  # YOUR CODE HERE

    val_mae = np.min(history.history['val_mae'])

    params = dict(
        context="train",
        training_set_size=DATA_SIZE,
        row_count=len(X_train_processed),
    )

    # Save results on the hard drive using taxifare.ml_logic.registry
    save_results(params=params, metrics=dict(mae=val_mae))

    # Save model weight on the hard drive (and optionally on GCS too!)
    save_model(model=model)

    print("✅ train() done \n")

    return val_mae

You'll need:
1. Build the query to the preprocessed table create in the earlier step
2. Pass that query to `ml_logic.data.get_data_with_cache()`. That function is already complete.
3. Manually split the preprocessed data into Train and test
4. Load the model using `ml_logic.registry.load_model()`.
5. Create a bucket manually in the console
6. Change the `BUCKET_NAME` accordingly in `.env`

It will look like this 👇

In [ ]:
def train(
        min_date:str = '2009-01-01',
        max_date:str = '2015-01-01',
        split_ratio: float = 0.02, # 0.02 represents ~ 1 month of validation data on a 2009-2015 train set
        learning_rate=0.0005,
        batch_size = 256,
        patience = 2
    ) -> float:

    """
    - Download processed data from your BQ table (or from cache if it exists)
    - Train on the preprocessed dataset (which should be ordered by date)
    - Store training results and model weights

    Return val_mae as a float
    """

    print(Fore.MAGENTA + "\n⭐️ Use case: train" + Style.RESET_ALL)
    print(Fore.BLUE + "\nLoading preprocessed validation data..." + Style.RESET_ALL)

    min_date = parse(min_date).strftime('%Y-%m-%d') # e.g '2009-01-01'
    max_date = parse(max_date).strftime('%Y-%m-%d') # e.g '2009-01-01'

    
    
    👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇
    👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇
    
    # Load processed data using `get_data_with_cache` in chronological order
    # Try it out manually on console.cloud.google.com first!
    
    # BUILD QUERY TO PREPROCESSED TABLE
    # Below, our columns are called ['_0', '_1'....'_66'] on BQ, student's column names may differ
    query = f"""
        SELECT * EXCEPT(_0)
        FROM {GCP_PROJECT}.{BQ_DATASET}.processed_{DATA_SIZE}
        WHERE _0 BETWEEN '{min_date}' AND '{max_date}'
        ORDER BY _0 ASC
    """

    data_processed_cache_path = Path(LOCAL_DATA_PATH).joinpath("processed", f"processed_{min_date}_{max_date}_{DATA_SIZE}.csv")
    data_processed = get_data_with_cache(
        gcp_project=GCP_PROJECT,
        query=query,
        cache_path=data_processed_cache_path,
        data_has_header=False
    )

    print(data_processed.shape)

    if data_processed.shape[0] < 10:
        print("❌ Not enough processed data retrieved to train on")
        return None

    # Create (X_train_processed, y_train, X_val_processed, y_val)
    train_length = int(len(data_processed)*(1-split_ratio))

    data_processed_train = data_processed.iloc[:train_length, :].sample(frac=1).to_numpy()
    data_processed_val = data_processed.iloc[train_length:, :].sample(frac=1).to_numpy()

    X_train_processed = data_processed_train[:, :-1]
    y_train = data_processed_train[:, -1]

    X_val_processed = data_processed_val[:, :-1]
    y_val = data_processed_val[:, -1]

    # Train model using `model.py`
    model = load_model()

    if model is None:
        model = initialize_model(input_shape=X_train_processed.shape[1:])

    model = compile_model(model, learning_rate=learning_rate)
    model, history = train_model(
        model, X_train_processed, y_train,
        batch_size=batch_size,
        patience=patience,
        validation_data=(X_val_processed, y_val)
    )

    print(history)

    val_mae = np.min(history.history['val_mae'])

    params = dict(
        context="train",
        training_set_size=DATA_SIZE,
        row_count=len(X_train_processed),
    )

    # Save results on the hard drive using taxifare.ml_logic.registry
    save_results(params=params, metrics=dict(mae=val_mae))

    # Save model weight on the hard drive (and optionally on GCS too!)
    save_model(model=model)

    print("✅ train() done \n")

    return val_mae

At this stage, you should be able to run `make run_train`, and see a model being saved in your bucket. 

If you do, you're done this step 🥳🥳🥳

## `make run_evaluate`

All you need to do here is:
1. Add the query 
2. Query using `get_data_with_cache()`

In [ ]:
def evaluate(
        min_date:str = '2014-01-01',
        max_date:str = '2015-01-01',
        stage: str = "Production"
    ) -> float:
    """
    Evaluate the performance of the latest production model on processed data
    Return MAE as a float
    """
    print(Fore.MAGENTA + "\n⭐️ Use case: evaluate" + Style.RESET_ALL)

    model = load_model(stage=stage)
    assert model is not None

    min_date = parse(min_date).strftime('%Y-%m-%d') # e.g '2009-01-01'
    max_date = parse(max_date).strftime('%Y-%m-%d') # e.g '2009-01-01'

    
    👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇
    👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇
    👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇👇
    
    # Query your BigQuery processed table and get data_processed using `get_data_with_cache`
    # Below, our columns are called ['_0', '_1'....'_66'] on BQ, student's column names may differ
    query = f"""
        SELECT * EXCEPT(_0)
        FROM {GCP_PROJECT}.{BQ_DATASET}.processed_{DATA_SIZE}
        WHERE _0 BETWEEN '{min_date}' AND '{max_date}'
        ORDER BY _0 ASC
    """

    data_processed_cache_path = Path(LOCAL_DATA_PATH).joinpath("processed", f"processed_{min_date}_{max_date}_{DATA_SIZE}.csv")
    data_processed = get_data_with_cache(
        gcp_project=GCP_PROJECT,
        query=query,
        cache_path=data_processed_cache_path,
        data_has_header=False)
    
    👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆
    👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆
    👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆👆

    if data_processed.shape[0] == 0:
        print("❌ No data to evaluate on")
        return None

    data_processed = data_processed.to_numpy()

    X_new = data_processed[:, :-1]
    y_new = data_processed[:, -1]

    metrics_dict = evaluate_model(model=model, X=X_new, y=y_new)
    mae = metrics_dict["mae"]

    params = dict(
        context="evaluate", # Package behavior
        training_set_size=DATA_SIZE,
        row_count=len(X_new)
    )
    
    👇👇👇
    save_results(params=params, metrics=metrics_dict)
    👆👆👆
    
    print("✅ evaluate() done \n")

    return mae

Note that the function `taxifare.ml_logic.registry.save_results()` always saves locally to the path located in the variable `LOCAL_REGISTRY_PATH` in the `taxifare.params` file.

## `make run_pred` 

Nothing to do here normally!!